# Tarea grupal: "EnviRápido", centro de distribución de paquetes


**Grupos de 5 · Lenguaje: Python

Este cuaderno contiene el **enunciado**, los **mínimos necesarios**, los **puntos de libertad**, los **entregables** y la **rúbrica**. Al final hay un espacio de trabajo con una celda de código por cada mínimo.



---
# 1. El problema
Una empresa de mensajería, **EnviRápido**, tiene un centro de distribución que hoy funciona con cuadernos y hojas de papel. Su dueño sufre estos problemas:

- Cada día llegan cientos de paquetes y se atienden **en el orden en que llegaron**, pero a veces los empleados se saltan paquetes sin darse cuenta.
- Los clientes llaman preguntando *"¿dónde está mi paquete?"* y el empleado tarda minutos en encontrarlo revisando hoja por hoja.
- A veces un empleado marca un paquete como "entregado" por error y no hay forma de corregirlo ordenadamente.
- La empresa reparte a **zonas/sucursales** (cientos de ellas, con un código numérico) y necesita saber rápidamente a cuál corresponde cada código.
- Hay paquetes que llegan **sin código de seguimiento**, solo con el nombre del cliente, y también hay que poder ubicarlos.

**Tu grupo es una empresa de software** que debe ofrecerle al dueño un prototipo que resuelva estos problemas. Ustedes deciden qué datos tiene un paquete, cómo se organiza el sistema y qué estructuras usar. El dueño no sabe de programación: tendrán que **convencerlo** de que su solución es buena y de por qué.

---
# 2. Mínimos necesarios (para aprobar)

1. **Modelado propio:** definir qué es un paquete, un cliente y una zona (clases o estructuras), y explicar por escrito qué decidieron y por qué.
2. **Recepción y atención en orden de llegada**, con posibilidad de ver quién sigue.
3. **Consulta rápida por código de seguimiento**, usando **una tabla hash implementada por ustedes** (sin usar `dict` ni `set` de Python para la estructura principal), con **manejo de colisiones**.
4. **Corrección de errores** (deshacer acciones, al menos las últimas) usando una estructura adecuada.
5. **Búsqueda de zona por código** sobre una lista ordenada, usando búsqueda binaria.
6. **Búsqueda de paquetes por nombre de cliente** (sin código), usando búsqueda secuencial.
7. **Pruebas:** un programa de ejemplo con al menos 20 paquetes que demuestre cada funcionalidad, incluyendo casos límite (cola vacía, código inexistente, colisión, deshacer sin historial).
8. **Medición:** comparar cuántas comparaciones hace cada búsqueda (secuencial vs. binaria vs. hash) con 100, 1000 y 10000 datos, y mostrar los resultados.

---
# 3. Puntos de libertad (donde tienen que pensar)
Estas decisiones **no las da el enunciado**; deben tomarlas y justificarlas:

- ¿Qué pasa con los paquetes urgentes o con prioridad? ¿Rompen la regla del orden de llegada?
- ¿Qué se hace si dos paquetes tienen el mismo nombre de cliente?
- ¿Cuál es el tamaño de la tabla hash y qué función hash usan? ¿Qué hacen cuando se llena?
- ¿Qué acciones se pueden deshacer y cuáles no?
- ¿La lista de zonas puede cambiar? Si cambia, ¿cómo se mantiene ordenada?
- ¿Cómo reportan al usuario que algo no se encontró?

**Extras opcionales (para nota adicional):** reasignar paquetes entre zonas, estadísticas del día, interfaz de menú por consola, carga de datos desde archivo.

---
# 4. Entregables y "venta"

| Entregable | Detalle |
|---|---|
| Código | Este cuaderno `.ipynb` (o archivos `.py`), comentado y ordenado |
| **Presentación de venta (10 min )** | Demostración en vivo + argumento de por qué su solución es la mejor |

**Reglas de la "venta":** los 5 integrantes deben hablar. El público (el resto de la clase y el docente) actúa como el dueño de la empresa y puede hacer preguntas como *"¿qué pasa si hay 1 millón de paquetes?"* o *"¿por qué no usaron una lista para esto?"*.



---
# ESPACIO DE TRABAJO DEL GRUPO

### Equipo y roles
Completen esta tabla (haciendo doble clic en la celda). Los roles deben **rotar** al preparar la defensa.

| Integrante | Rol principal | Rol de la defensa |
|---|---|---|
| 1 Daniela Fernandez| Minimo 2,3 | Presentacion, Defensa de Solucion, Atencion, Almacenamiento |
| 2 David Diaz| Minimo 4, 5 |Defensa de Deshacer Acciones, Busqueda |
| 3 Diego Morales| Minimo 1, 8 | Conclusiones y Cierre |
| 4 Josue Vargas| Minimo 6, 7 | Defensa de Pruebas y Comparaciones  |
| 5 | | |

### Registro de decisiones de diseño
Escriban aquí, en pocas líneas, qué decidieron para cada punto de libertad y **por qué** (esto alimenta el informe).

- **Paquetes urgentes:**
- **Nombres de cliente repetidos:**
- **Tamaño y función hash / tabla llena:**
- **Qué se puede deshacer y qué no:**
- **Zonas (¿cambian? ¿cómo se mantienen ordenadas?):**
- **Cómo se informa que algo no se encontró:**

### Preparación: módulos
Importen aquí lo que necesiten (por ejemplo `randint` para generar datos de prueba). **Recuerden:** no usar `dict` ni `set` para la tabla hash principal.

In [13]:
from random import randint  # agreguen aquí los módulos que necesiten
import hashlib
from collections import deque


### Mínimo 1 · Modelado: paquete, cliente y zona
Definan las clases o estructuras. Antes de programar, escriban en papel los **sustantivos** y los **verbos** del problema.

In [14]:
class Cliente:
    def __init__(self, nombre):
        self.nombre = nombre
        self.paquetes = []

    def agregar_paquete(self, paquete):
        self.paquetes.append(paquete)

    def eliminar_paquete(self, paquete):
        if paquete in self.paquetes:
            self.paquetes.remove(paquete)

    def __str__(self):
        return f"Cliente: {self.nombre} (Paquetes: {len(self.paquetes)})"

In [15]:
class Zona:
  def __init__(self, codigo, nombre):
    self.codigo = codigo
    self.nombre = nombre
  def __str__(self):
    return f"Zona {self.codigo} - Nombre: {self.nombre}"

In [16]:
class Paquete:
  def __init__(self, codigo, cliente, zona):
    self.codigo = codigo
    self.cliente = cliente
    self.estado = 'Recibido'
    self.zona = zona

    if cliente is not None:
      cliente.agregar_paquete(self)

  def actualizar_estado(self, nuevo_estado):
    self.estado = nuevo_estado

  def __str__(self):
    return f"Paquete {self.codigo} - Cliente: {self.cliente.nombre} - Estado: {self.estado} - Zona: {self.zona.nombre}"

### Mínimo 2 · Recepción y atención en orden de llegada
Necesitan: recibir un paquete, atender el siguiente y **ver quién sigue** sin sacarlo.

In [17]:
class ColaPaquetes:
    def __init__(self):
        self.items = []

    def encolar(self, paquete):
        """Agrega un paquete al final de la cola (recepción)."""
        self.items.append(paquete)

    def desencolar(self):
        """Atiende y elimina el primer paquete que llegó (atención)."""
        if self.esta_vacia():
            return None
        return self.items.pop(0)

    def ver_siguiente(self):
        """Muestra el siguiente paquete en la fila sin atenderlo (peek)."""
        if self.esta_vacia():
            return None
        return self.items[0]

    def esta_vacia(self):
        """Verifica si la cola no tiene elementos."""
        return len(self.items) == 0

    def tamaño(self):
        """Devuelve el número de paquetes en espera."""
        return len(self.items)

    def __str__(self):
        if self.esta_vacia():
            return "Cola vacía"
        return " -> ".join([str(p.codigo) for p in self.items])

### Mínimo 3 · Consulta por código de seguimiento (tabla hash propia)
Implementen la tabla hash completa: función hash, insertar, buscar y manejo de colisiones. **Sin `dict` ni `set`.**

In [18]:
class TablaHash:
    def __init__(self, capacidad=50):
        self.capacidad = capacidad
        self.tabla = [[] for _ in range(capacidad)]

    def funcion_hash(self, codigo):
      #solo funciona si el codigo es int, le voy a meter una funcion para si es str por un caso
        if isinstance(codigo, str):
          codigo = sum(ord(char) for char in codigo)
        return codigo % self.capacidad

    def insertar(self, paquete):
        indice = self.funcion_hash(paquete.codigo)

        for i, (codigo, paquete_existente) in enumerate(self.tabla[indice]):
            if codigo == paquete.codigo:
                self.tabla[indice][i] = (paquete.codigo, paquete)
                return
        self.tabla[indice].append((paquete.codigo, paquete))

    def buscar(self, codigo):
        indice = self.funcion_hash(codigo)
        for codigo_existente, paquete in self.tabla[indice]:
            if codigo_existente == codigo:
                return paquete
        return None

### Mínimo 4 · Deshacer acciones
Decidan qué acciones se registran y **qué información hay que guardar** para poder revertirlas.

In [19]:
# Tu código aquí: historial y deshacer
# Minimo 4 - Deshacer acciones usando una pila

class HistorialAcciones:
    def __init__(self, cola, tabla_hash):
        self.pila = []
        self.cola = cola
        self.tabla_hash = tabla_hash

    def recibir_paquete(self, paquete):
        # Registramos el paquete en la cola y en la tabla hash y al final lo guardamos
        self.cola.encolar(paquete)
        self.tabla_hash.insertar(paquete)
        self.pila.append(("recibir", paquete))

    def atender_paquete(self):
        paquete = self.cola.desencolar()

        if paquete is None:
            print("No hay paquetes para atender")
            return None

        # Guardamos el estado anterior
        estado_anterior = paquete.estado
        paquete.actualizar_estado("En reparto")
        self.pila.append(("atender", paquete, estado_anterior))
        return paquete

    def cambiar_estado(self, paquete, nuevo_estado):
        estado_anterior = paquete.estado
        paquete.actualizar_estado(nuevo_estado)
        self.pila.append(("estado", paquete, estado_anterior))

    def deshacer(self):
        if len(self.pila) == 0:
            print("No hay acciones para deshacer")
            return False

        accion = self.pila.pop()
        tipo = accion[0]
        paquete = accion[1]

        if tipo == "recibir":
            self.cola.items.remove(paquete)
            paquete.cliente.eliminar_paquete(paquete)

            # Eliminamos el paquete de la tabla hash
            indice = self.tabla_hash.funcion_hash(paquete.codigo)
            cubeta = self.tabla_hash.tabla[indice]

            for i, (codigo, guardado) in enumerate(cubeta):
                if guardado is paquete:
                    cubeta.pop(i)
                    break

            print("Recepcion deshecha:", paquete.codigo)

        elif tipo == "atender":
            # Devolvemos el paquete al principio de la cola
            self.cola.items.insert(0, paquete)
            paquete.actualizar_estado(accion[2])
            print("Atencion deshecha:", paquete.codigo)

        elif tipo == "estado":
            paquete.actualizar_estado(accion[2])
            print("Cambio de estado deshecho:", paquete.codigo)

        return True

### Mínimo 5 · Búsqueda de zona por código (búsqueda binaria)
La lista de zonas debe estar **ordenada** antes de buscar.

In [20]:
# Tu código aquí: lista de zonas ordenada y búsqueda binaria
# Minimo 5 - Busqueda binaria de zonas

class BusquedaZonas:
    def __init__(self):
        self.zonas = []

    def agregar_zona(self, zona):
        # Evitamos codigos repetidos
        for existente in self.zonas:
            if existente.codigo == zona.codigo:
                return False
        self.zonas.append(zona)
        #esto le puse para que se mantenga ordenado la parte de codigos
        self.zonas.sort(key=lambda z: z.codigo)
        return True

    def busqueda_binaria(self, codigo):
        izquierda = 0
        derecha = len(self.zonas) - 1
        comparaciones = 0

        while izquierda <= derecha:
            medio = (izquierda + derecha) // 2
            comparaciones += 1
            if self.zonas[medio].codigo == codigo:
                return self.zonas[medio], comparaciones

            elif self.zonas[medio].codigo < codigo:
                izquierda = medio + 1

            else:
                derecha = medio - 1

        return None, comparaciones

### Mínimo 6 · Búsqueda de paquetes por nombre de cliente (búsqueda secuencial)
Piensen qué devuelve su búsqueda si hay varios paquetes con el mismo nombre.

In [21]:
# Decision: si varios paquetes tienen el mismo nombre de cliente,
# devolvemos TODOS en una lista (no solo el primero), porque un cliente
# puede tener varios envios o puede haber dos clientes con el mismo nombre.
# La busqueda no distingue mayusculas/minusculas ni espacios extra.

def buscar_por_cliente(lista_paquetes, nombre):
    nombre_buscado = nombre.strip().lower()
    encontrados = []
    comparaciones = 0

    # Recorremos la lista de principio a fin (busqueda secuencial)
    for paquete in lista_paquetes:
        comparaciones += 1
        if paquete.cliente.nombre.strip().lower() == nombre_buscado:
            encontrados.append(paquete)

    return encontrados, comparaciones


def mostrar_busqueda_cliente(lista_paquetes, nombre):
    encontrados, comparaciones = buscar_por_cliente(lista_paquetes, nombre)
    if len(encontrados) == 0:
        print(f"No se encontraron paquetes para el cliente '{nombre}' ({comparaciones} comparaciones)")
    else:
        print(f"{len(encontrados)} paquete(s) de '{nombre}' ({comparaciones} comparaciones):")
        for p in encontrados:
            print("  ", p)
    return encontrados

### Mínimo 7 · Pruebas con al menos 20 paquetes
Demuestren **cada** funcionalidad. Incluyan casos límite: cola vacía, código inexistente, colisión y deshacer sin historial.

In [22]:

print("=" * 60)
print("PREPARACION: zonas, clientes y estructuras")
print("=" * 60)

# Zonas (se agregan desordenadas a proposito para probar que se ordenan)
busqueda_zonas = BusquedaZonas()
datos_zonas = [(40, "Sur"), (10, "Norte"), (30, "Este"), (20, "Centro"), (50, "Oeste")]
for codigo, nombre in datos_zonas:
    busqueda_zonas.agregar_zona(Zona(codigo, nombre))
print("Zonas ordenadas:", [z.codigo for z in busqueda_zonas.zonas])
print("Agregar zona repetida (10):", busqueda_zonas.agregar_zona(Zona(10, "Norte 2")))

# Clientes (Ana Lopez aparece dos veces a proposito: mismo nombre, dos clientes)
nombres = ["Ana Lopez", "Carlos Ruiz", "Maria Perez", "Jose Vera", "Ana Lopez",
           "Luis Mora", "Sofia Cano", "Pedro Gil"]
clientes = [Cliente(n) for n in nombres]

cola = ColaPaquetes()
tabla = TablaHash(capacidad=50)
historial = HistorialAcciones(cola, tabla)
registro = []   # lista con todos los paquetes recibidos (para busqueda secuencial)

print()
print("=" * 60)
print("PRUEBA 1: casos limite con todo vacio")
print("=" * 60)
print("Siguiente en cola vacia:", cola.ver_siguiente())
print("Atender con cola vacia:", historial.atender_paquete())
print("Deshacer sin historial:", historial.deshacer())

print()
print("=" * 60)
print("PRUEBA 2: recepcion de 22 paquetes (orden de llegada)")
print("=" * 60)
for i in range(22):
    codigo = 1001 + i
    cliente = clientes[i % len(clientes)]
    zona = busqueda_zonas.zonas[i % len(busqueda_zonas.zonas)]
    paquete = Paquete(codigo, cliente, zona)
    historial.recibir_paquete(paquete)
    registro.append(paquete)
print("Paquetes recibidos:", len(registro))
print("Cola:", cola)
print("Tamaño de la cola:", cola.tamaño())

print()
print("=" * 60)
print("PRUEBA 3: atencion en orden de llegada y ver quien sigue")
print("=" * 60)
print("Siguiente (sin sacarlo):", cola.ver_siguiente())
atendido = historial.atender_paquete()
print("Atendido:", atendido)
atendido = historial.atender_paquete()
print("Atendido:", atendido)
print("Ahora sigue:", cola.ver_siguiente().codigo)

print()
print("=" * 60)
print("PRUEBA 4: consulta por codigo (tabla hash)")
print("=" * 60)
print("Buscar 1010:", tabla.buscar(1010))
print("Buscar 9999 (inexistente):", tabla.buscar(9999))

# Colision: con capacidad 50, 1051 % 50 = 1 y 1001 % 50 = 1 -> misma cubeta
p_colision = Paquete(1051, clientes[1], busqueda_zonas.zonas[0])
historial.recibir_paquete(p_colision)
registro.append(p_colision)
indice = tabla.funcion_hash(1051)
print(f"Colision: 1001 y 1051 van a la cubeta {tabla.funcion_hash(1001)} y {indice}")
print("Contenido de esa cubeta:", [codigo for codigo, _ in tabla.tabla[indice]])
print("Buscar 1001:", tabla.buscar(1001).codigo, "| Buscar 1051:", tabla.buscar(1051).codigo)

# Codigo de tipo texto
p_texto = Paquete("ENV-77", clientes[2], busqueda_zonas.zonas[2])
historial.recibir_paquete(p_texto)
registro.append(p_texto)
print("Buscar 'ENV-77':", tabla.buscar("ENV-77"))

print()
print("=" * 60)
print("PRUEBA 5: deshacer acciones")
print("=" * 60)
historial.deshacer()   # deshace la recepcion de ENV-77
registro.remove(p_texto)
print("Buscar 'ENV-77' despues de deshacer:", tabla.buscar("ENV-77"))

paquete_x = tabla.buscar(1005)
historial.cambiar_estado(paquete_x, "Entregado")
print("Estado de 1005:", paquete_x.estado)
historial.deshacer()   # deshace el cambio de estado
print("Estado de 1005 despues de deshacer:", paquete_x.estado)

print("Cola antes:", cola)
historial.atender_paquete()
print("Cola despues de atender:", cola)
historial.deshacer()   # el paquete vuelve al frente
print("Cola despues de deshacer:", cola)

print()
print("=" * 60)
print("PRUEBA 6: busqueda binaria de zonas")
print("=" * 60)
zona, comp = busqueda_zonas.busqueda_binaria(30)
print("Buscar zona 30:", zona, f"({comp} comparaciones)")
zona, comp = busqueda_zonas.busqueda_binaria(99)
print("Buscar zona 99 (inexistente):", zona, f"({comp} comparaciones)")

print()
print("=" * 60)
print("PRUEBA 7: busqueda secuencial por nombre de cliente")
print("=" * 60)
mostrar_busqueda_cliente(registro, "Ana Lopez")      # nombre repetido
mostrar_busqueda_cliente(registro, "  pedro gil ")   # mayusculas y espacios
mostrar_busqueda_cliente(registro, "Juan Nadie")     # inexistente

print()
print("=" * 60)
print("PRUEBA 8: vaciar la cola por completo")
print("=" * 60)
while not cola.esta_vacia():
    historial.atender_paquete()
print("Cola:", cola)
print("Atender con cola vacia:", historial.atender_paquete())
print()
print("Todas las pruebas terminaron.")

PREPARACION: zonas, clientes y estructuras
Zonas ordenadas: [10, 20, 30, 40, 50]
Agregar zona repetida (10): False

PRUEBA 1: casos limite con todo vacio
Siguiente en cola vacia: None
No hay paquetes para atender
Atender con cola vacia: None
No hay acciones para deshacer
Deshacer sin historial: False

PRUEBA 2: recepcion de 22 paquetes (orden de llegada)
Paquetes recibidos: 22
Cola: 1001 -> 1002 -> 1003 -> 1004 -> 1005 -> 1006 -> 1007 -> 1008 -> 1009 -> 1010 -> 1011 -> 1012 -> 1013 -> 1014 -> 1015 -> 1016 -> 1017 -> 1018 -> 1019 -> 1020 -> 1021 -> 1022
Tamaño de la cola: 22

PRUEBA 3: atencion en orden de llegada y ver quien sigue
Siguiente (sin sacarlo): Paquete 1001 - Cliente: Ana Lopez - Estado: Recibido - Zona: Norte
Atendido: Paquete 1001 - Cliente: Ana Lopez - Estado: En reparto - Zona: Norte
Atendido: Paquete 1002 - Cliente: Carlos Ruiz - Estado: En reparto - Zona: Centro
Ahora sigue: 1003

PRUEBA 4: consulta por codigo (tabla hash)
Buscar 1010: Paquete 1010 - Cliente: Carlos Ru

### Mínimo 8 · Medición de comparaciones
Midan cuántas comparaciones hace cada búsqueda (secuencial, binaria y hash) con **100, 1000 y 10000** datos y muestren una tabla con los resultados. Después escriban en la celda siguiente qué observan.

In [23]:
def secuencial(lista, codigo):
    comp = 0
    for p in lista:
        comp += 1
        if p.codigo == codigo:
            break
    return comp

def binaria(lista, codigo):
    izq = 0
    der = len(lista) - 1
    comp = 0
    while izq <= der:
        medio = (izq + der) // 2
        comp += 1
        if lista[medio].codigo == codigo:
            break
        elif lista[medio].codigo < codigo:
            izq = medio + 1
        else:
            der = medio - 1
    return comp

def con_hash(tabla, codigo):
    comp = 0
    indice = tabla.funcion_hash(codigo)
    for c, p in tabla.tabla[indice]:
        comp += 1
        if c == codigo:
            break
    return comp

cli = Cliente("Prueba")
zon = Zona(10, "Norte")

print("Datos | Secuencial | Binaria | Hash")
for n in [100, 1000, 10000]:
    lista = []
    tabla = TablaHash(capacidad=50)
    for i in range(n):
        p = Paquete(i, cli, zon)
        lista.append(p)
        tabla.insertar(p)
    ultimo = n - 1  # buscamos el ultimo paquete
    print(n, "|", secuencial(lista, ultimo), "|", binaria(lista, ultimo), "|", con_hash(tabla, ultimo))

Datos | Secuencial | Binaria | Hash
100 | 100 | 7 | 2
1000 | 1000 | 10 | 20
10000 | 10000 | 14 | 200


### Análisis de las mediciones

Al multiplicar por 10 la cantidad de paquetes (de 100 a 1.000 y luego a 10.000), observamos que las tres búsquedas aumentan sus comparaciones de manera diferente:

| Cantidad de datos | Secuencial | Binaria | Hash |
|---:|---:|---:|---:|
| 100 | 100 | 7 | 2 |
| 1.000 | 1.000 | 10 | 20 |
| 10.000 | 10.000 | 14 | 200 |

- **Búsqueda secuencial:** creció de forma proporcional a los datos: al multiplicarlos por 10, las comparaciones también se multiplicaron por 10. Esto ocurre porque buscamos el último paquete recorriendo la lista desde el inicio. Su complejidad es **O(n)**.
- **Búsqueda binaria:** aumentó muy lentamente (7, 10 y 14 comparaciones), porque en cada iteración descarta aproximadamente la mitad de la lista ordenada. Su complejidad es **O(log n)** y fue la más eficiente en esta prueba concreta.
- **Tabla hash:** pasó de 2 a 20 y luego a 200 comparaciones. Aunque normalmente una tabla hash ofrece búsquedas **O(1) en promedio** con una buena distribución y capacidad adecuada, nuestra implementación mantiene **50 cubetas fijas**. A medida que crecen los datos, aumenta la cantidad de paquetes por cubeta (colisiones por encadenamiento), y la búsqueda se vuelve más costosa.

**Conclusión:** la búsqueda secuencial creció más rápido en cantidad total de comparaciones, mientras que la binaria mostró el crecimiento más lento. La tabla hash no mantuvo el rendimiento constante esperado porque su capacidad no aumentó con los registros. Como mejora futura proponemos **redimensionar la tabla y recalcular los índices** cuando el factor de carga sea elevado. Estas mediciones comparan búsquedas por código y cuentan comparaciones de claves; no incluyen el costo de construir estructuras, calcular el hash u ordenar los datos.

### Lista de verificación antes de entregar
- [ ] Los 8 mínimos funcionan y están demostrados
- [ ] La tabla hash no usa `dict` ni `set` y maneja colisiones
- [ ] El código está comentado y ordenado
- [ ] Los 5 integrantes pueden explicar **cualquier** parte, no solo la suya
